# LeetCode #1135: Connecting Cities With Minimum Cost

https://leetcode.com/problems/connecting-cities-with-minimum-cost/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(V^E)$ | $O(V)$ |
| **Optimal: Kruskal's MST (Union-Find) ★** | $O(E \log E)$ | $O(V)$ |

---

## Understanding the Methods

### Brute Force
Try all subsets of edges to find the minimum-cost spanning tree. The exponential number of subsets makes this completely infeasible for large graphs.

### Optimal: Kruskal's MST (Union-Find) ★
Sort all edges by cost, then greedily add the cheapest edge that connects two different components (detected via Union-Find path compression). Stop when all $V$ cities are in one component. If fewer than $V-1$ edges were added, the graph is disconnected — return $-1$.

**Constraints:**
* $1 \leq n \leq 10^4$ (cities)
* $1 \leq \text{connections.length} \leq 10^4$
* `connections[i] = [city1, city2, cost]`
* $1 \leq \text{city1}, \text{city2} \leq n$, city1 $\neq$ city2

## Solutions

### C#

In [ ]:
public class Solution {
    private int[] _parent, _rank;

    public int MinimumCost(int n, int[][] connections) {
        _parent = Enumerable.Range(0, n + 1).ToArray();
        _rank = new int[n + 1];
        // Process cheapest edges first so every accepted edge minimizes total cost
        Array.Sort(connections, (a, b) => a[2] - b[2]);
        int totalCost = 0, edgesUsed = 0;
        foreach (var conn in connections) {
            if (Union(conn[0], conn[1])) {
                totalCost += conn[2];
                // Exactly n-1 edges needed to span n cities
                if (++edgesUsed == n - 1) return totalCost;
            }
        }
        return -1; // graph is disconnected
    }

    private int Find(int x) =>
        _parent[x] == x ? x : (_parent[x] = Find(_parent[x]));

    private bool Union(int x, int y) {
        int px = Find(x), py = Find(y);
        if (px == py) return false;
        // Attach smaller rank tree under larger to keep the union-find flat
        if (_rank[px] < _rank[py]) (px, py) = (py, px);
        _parent[py] = px;
        if (_rank[px] == _rank[py]) _rank[px]++;
        return true;
    }
}

### Python

In [ ]:
class Solution:
    def minimum_cost(self, n: int, connections: list[list[int]]) -> int:
        parent = list(range(n + 1))
        rank = [0] * (n + 1)

        def find(x: int) -> int:
            # Path compression flattens the tree for near-O(1) future lookups
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        def union(x: int, y: int) -> bool:
            px, py = find(x), find(y)
            if px == py: return False
            # Attach smaller rank tree under larger to keep the union-find flat
            if rank[px] < rank[py]: px, py = py, px
            parent[py] = px
            if rank[px] == rank[py]: rank[px] += 1
            return True

        # Process cheapest edges first so every accepted edge minimizes total cost
        connections.sort(key=lambda c: c[2])
        total, edges = 0, 0
        for c1, c2, cost in connections:
            if union(c1, c2):
                total += cost
                edges += 1
                # Exactly n-1 edges needed to span n cities
                if edges == n - 1: return total
        return -1

### Go

In [ ]:
import "sort"

func minimumCost(n int, connections [][]int) int {
    parent := make([]int, n+1)
    rankArr := make([]int, n+1)
    for i := range parent { parent[i] = i }

    var find func(x int) int
    find = func(x int) int {
        // Path compression flattens the tree for near-O(1) future lookups
        for parent[x] != x { parent[x] = parent[parent[x]]; x = parent[x] }
        return x
    }
    union := func(x, y int) bool {
        px, py := find(x), find(y)
        if px == py { return false }
        // Attach smaller rank tree under larger to keep the union-find flat
        if rankArr[px] < rankArr[py] { px, py = py, px }
        parent[py] = px
        if rankArr[px] == rankArr[py] { rankArr[px]++ }
        return true
    }

    // Process cheapest edges first so every accepted edge minimizes total cost
    sort.Slice(connections, func(i, j int) bool { return connections[i][2] < connections[j][2] })
    total, edges := 0, 0
    for _, c := range connections {
        if union(c[0], c[1]) {
            total += c[2]; edges++
            // Exactly n-1 edges needed to span n cities
            if edges == n-1 { return total }
        }
    }
    return -1
}

### Rust

In [ ]:
impl Solution {
    pub fn minimum_cost(n: i32, mut connections: Vec<Vec<i32>>) -> i32 {
        let n = n as usize;
        let mut parent: Vec<usize> = (0..=n).collect();
        let mut rank = vec![0usize; n + 1];

        fn find(parent: &mut Vec<usize>, mut x: usize) -> usize {
            // Path compression flattens the tree for near-O(1) future lookups
            while parent[x] != x { parent[x] = parent[parent[x]]; x = parent[x]; }
            x
        }

        // Process cheapest edges first so every accepted edge minimizes total cost
        connections.sort_by_key(|c| c[2]);
        let mut total = 0i32;
        let mut edges = 0usize;
        for c in &connections {
            let (mut px, mut py) = (find(&mut parent, c[0] as usize), find(&mut parent, c[1] as usize));
            if px == py { continue; }
            // Attach smaller rank tree under larger to keep the union-find flat
            if rank[px] < rank[py] { std::mem::swap(&mut px, &mut py); }
            parent[py] = px;
            if rank[px] == rank[py] { rank[px] += 1; }
            total += c[2]; edges += 1;
            // Exactly n-1 edges needed to span n cities
            if edges == n - 1 { return total; }
        }
        -1
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `n = 3`, `connections = [[1,2,5],[1,3,6],[2,3,1]]`
Sort by cost: $[[2,3,1],[1,2,5],[1,3,6]]$. Add edge $(2,3)$ cost $1$, then $(1,2)$ cost $5$ — two edges span 3 cities. Total = $6$.

### 2. Slightly Complex
**Input:** `n = 4`, `connections = [[1,2,3],[3,4,4],[1,4,2],[1,3,6]]`
Sorted: $(1,4,2),(1,2,3),(3,4,4)$. Adding all three spans 4 cities with cost $9$. The edge $(1,3,6)$ is never needed.

### 3. Edge Case: Time Factor
**Input:** $10^4$ cities, $10^4$ edges, sorted already
Sorting is $O(E \log E) = O(10^4 \times 14) \approx 140{,}000$ operations. Union-Find with path compression processes each edge in nearly $O(1)$. Full scan without early exit if the MST spans all $n-1$ edges.

### 4. Edge Case: Space Factor
**Input:** `n = 2`, `connections = [[1,2,100]]`
Minimal graph: one edge connects two cities. Union-Find arrays have 3 entries. Total cost $= 100$ — one edge gives the MST in $O(1)$ after sorting.

### 5. Almost-Impossible but Plausible
**Input:** `n = 5`, `connections = [[1,2,1],[3,4,1],[1,3,1],[2,4,1]]` — 4 edges, city 5 isolated
After processing all 4 edges we have only $4$ edges used out of the $5-1=4$ needed, but city $5$ is unreachable. The function returns $-1$, confirming the disconnection check fires correctly even when the other 4 cities form a spanning tree.